# Assignment 2: Diabetes Classification

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('diabetes_binary_classification_data.csv')

# All values are whole numbers stored as floats, so we store them as integers
df = df.astype(int)

print(df.shape)
df.head()

# Step 1 - Understand the data

## Features and target

- **Target (y):** `Diabetes_binary`
- **Input features (X):** the other 21 columns

In [ ]:
X = df.drop(columns=['Diabetes_binary'])
y = df['Diabetes_binary']

print('X:', X.shape)
print('y:', y.shape)
print('Missing values:', df.isna().sum().sum())

## What do class 0 and class 1 mean?

- **0** = no diabetes
- **1** = prediabetes or diabetes

Class 1 combines prediabetes and diabetes, so the model cannot tell them apart.

## How many observations are in each class?

In [ ]:
class_counts = pd.DataFrame({
    'Count': y.value_counts(),
    'Proportion': y.value_counts(normalize=True).round(3),
})
class_counts

The target is **imbalanced**: only about 14% are class 1. A model that always predicts "no diabetes" would already be about 86% accurate, so accuracy alone is not enough to judge the model. We also need precision and recall.

## Three features of different types

All columns are stored as numbers, but the numbers mean different things. We use the *Understanding the Dataset* guide to read them:

| Feature | Type | What the values mean |
|---|---|---|
| `BMI` | Numerical | Body Mass Index, a real measurement (12-98) |
| `HighBP` | Binary | 0 = no high blood pressure, 1 = high blood pressure |
| `GenHlth` | Ordered | Self-rated health: 1 = excellent, 2 = very good, 3 = good, 4 = fair, 5 = poor |

In [ ]:
print(df['BMI'].describe().round(1), '\n')
print(df['HighBP'].value_counts().sort_index(), '\n')
print(df['GenHlth'].value_counts().sort_index())

- `BMI` is a real amount, so a higher number means a higher BMI.
- `HighBP` is a yes/no answer, so 0 and 1 are two categories, not amounts.
- `GenHlth` has an order (higher = worse health), but the steps between codes are not measured distances. A 4 is not "twice as bad" as a 2.

So looking only at the numbers in the CSV is not enough. The same kind of number can be a measurement, a category or an ordered group.

# Step 2 - Explore the data

We explore every feature against `Diabetes_binary`, using a method that fits each feature type:

- **Numerical:** compare the distributions for class 0 and class 1.
- **Binary:** compare the share with diabetes when the feature is 0 vs 1.
- **Ordered:** see how the share with diabetes changes across the categories.

EDA shows associations in this data. It does not show that a feature causes diabetes.

In [ ]:
numerical = ['BMI', 'MentHlth', 'PhysHlth']
binary = ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
          'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare',
          'NoDocbcCost', 'DiffWalk', 'Sex']
ordered = ['GenHlth', 'Age', 'Education', 'Income']

# Colors: blue for class 0 / "no", orange for class 1 / "yes"
BLUE, ORANGE, GRID = '#2a78d6', '#e0742b', '#e5e5e3'
overall_rate = y.mean()
print(f'Share with diabetes overall: {overall_rate:.1%}')

## Numerical features

In [ ]:
df.groupby('Diabetes_binary')[numerical].describe().T.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
bins = {'BMI': range(12, 70), 'MentHlth': range(0, 32), 'PhysHlth': range(0, 32)}

for ax, col in zip(axes, numerical):
    for cls, color, label in [(0, BLUE, 'Class 0 (no diabetes)'), (1, ORANGE, 'Class 1 (diabetes)')]:
        ax.hist(df.loc[y == cls, col], bins=bins[col], density=True,
                histtype='step', linewidth=2, color=color, label=label)
    ax.set_title(col)
    ax.set_ylabel('Share of class')
    ax.set_axisbelow(True)
    ax.grid(axis='y', color=GRID)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].legend()
plt.tight_layout()
plt.show()

- `BMI` is shifted to the right for class 1 (median 31 vs 27).
- `MentHlth` and `PhysHlth` are mostly 0, with a spike at 30 (people who answered "every day"). `PhysHlth` differs more between the classes than `MentHlth`.
- BMI goes up to 98, but values above about 60 are rare (805 people), so we keep them.

The histograms show each class as a share of its own class, because class 1 is much smaller.

## Binary features

In [ ]:
binary_table = pd.DataFrame({
    'Share with feature = 1': df[binary].mean(),
    'Diabetes share when 0': [y[df[col] == 0].mean() for col in binary],
    'Diabetes share when 1': [y[df[col] == 1].mean() for col in binary],
})
binary_table['Difference'] = binary_table['Diabetes share when 1'] - binary_table['Diabetes share when 0']
binary_table = binary_table.sort_values('Difference')
binary_table.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
rows = range(len(binary_table))

ax.hlines(rows, binary_table['Diabetes share when 0'], binary_table['Diabetes share when 1'],
          color=GRID, linewidth=2)
ax.scatter(binary_table['Diabetes share when 0'], rows, s=60, color=BLUE, label='Feature = 0 (no)', zorder=3)
ax.scatter(binary_table['Diabetes share when 1'], rows, s=60, color=ORANGE, label='Feature = 1 (yes)', zorder=3)
ax.axvline(overall_rate, color='#52514e', linestyle='--', linewidth=1, label='Overall share')

ax.set_yticks(list(rows))
ax.set_yticklabels(binary_table.index)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_xlabel('Share with diabetes')
ax.set_title('Share with diabetes when each binary feature is 0 vs 1')
ax.set_axisbelow(True)
ax.grid(axis='x', color=GRID)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

- Health conditions show the biggest differences: people with heart disease, walking difficulty, a stroke, high blood pressure or high cholesterol have a much higher diabetes share.
- `CholCheck`: only 2.5% of people without a cholesterol check in the last 5 years have diabetes. This is probably because people with diabetes see a doctor more often, not because the check protects them.
- `PhysActivity`, `Veggies`, `Fruits` and `HvyAlcoholConsump` go the other way: the diabetes share is lower when the answer is "yes".
- `Sex`, `AnyHealthcare`, `NoDocbcCost` and `Smoker` show only small differences.
- Some features are very unbalanced. For example, 96% have had a cholesterol check and only 6% are heavy drinkers.

## Ordered features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.flat, ordered):
    rate = df.groupby(col)['Diabetes_binary'].mean()
    ax.bar(rate.index, rate.values, color=BLUE, width=0.7)
    ax.axhline(overall_rate, color='#52514e', linestyle='--', linewidth=1, label='Overall share')
    ax.set_title(col)
    ax.set_xticks(rate.index)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax.set_ylabel('Share with diabetes')
    ax.set_axisbelow(True)
    ax.grid(axis='y', color=GRID)
    ax.spines[['top', 'right']].set_visible(False)

axes[0, 0].legend()
plt.tight_layout()
plt.show()

for col in ordered:
    print(col, df[col].value_counts().sort_index().to_dict())

Codes are explained in the guide. For example, `GenHlth` 1 = excellent and 5 = poor; `Age` 1 = 18-24 and 13 = 80+.

- `GenHlth`: the diabetes share rises steadily from excellent to poor health.
- `Age`: the share rises with age up to about 70-74, then flattens.
- `Education` and `Income`: the share falls as education and income go up.
- Some categories are small, for example only 174 people have `Education` = 1.

## Main findings

**1. Is self-rated health related to diabetes?**
- *Evidence:* the `GenHlth` plot. The diabetes share goes from 2.5% (excellent) to 38% (poor).
- *Conclusion:* `GenHlth` has the clearest relationship with the target and looks useful for prediction.
- *We cannot conclude* that poor health causes diabetes. Having diabetes could also make people rate their health as worse.

**2. Are other health conditions related to diabetes?**
- *Evidence:* the binary plot. With high blood pressure the share is 24%, vs 6% without. Heart disease, stroke and walking difficulty show similar gaps.
- *Conclusion:* these conditions often appear together with diabetes in this data.
- *We cannot conclude* which one comes first, or that one causes another.

**3. Does BMI differ between the classes?**
- *Evidence:* the BMI histogram and the table below. The share with diabetes rises from about 6% for BMI under 25 to 34% for BMI over 40.
- *Conclusion:* higher BMI is associated with diabetes.

**4. Does the diabetes share change with age?**
- *Evidence:* the `Age` plot. It rises from about 1% in the youngest group to about 22% at ages 70-74.
- *Conclusion:* age is strongly associated with diabetes.

**5. Which features show little relationship, or a surprising one?**
- *Evidence:* `Sex`, `AnyHealthcare`, `NoDocbcCost` and `Smoker` differ by only 2-4 percentage points. Heavy drinkers have a *lower* diabetes share (6% vs 14%).
- *Conclusion:* these features probably add less to the model.
- *We cannot conclude* that alcohol protects against diabetes. Other differences between the groups, such as age, could explain the pattern.

In [ ]:
bmi_groups = pd.cut(df['BMI'], bins=[0, 18.5, 25, 30, 35, 40, 100],
                    labels=['<18.5', '18.5-25', '25-30', '30-35', '35-40', '40+'])
pd.DataFrame({
    'People': bmi_groups.value_counts().sort_index(),
    'Share with diabetes': df.groupby(bmi_groups, observed=True)['Diabetes_binary'].mean().round(3),
})

# Step 3 - Create TRAIN and TEST

- 80% TRAIN, 20% TEST.
- `stratify=y` keeps about the same share of class 1 (about 14%) in both sets.
- `random_state=42` gives the same split every run.

TEST is kept separate until Step 6. We do not balance or resample it.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pd.DataFrame({
    'Rows': [len(y_train), len(y_test)],
    'Class 0': [(y_train == 0).sum(), (y_test == 0).sum()],
    'Class 1': [(y_train == 1).sum(), (y_test == 1).sum()],
    'Share class 1': [y_train.mean().round(3), y_test.mean().round(3)],
}, index=['TRAIN', 'TEST'])

Both sets have the same share of class 1 as the full dataset (13.9%).

# Step 4 - Prepare the features

We looked at the guide to decide how to handle each type of feature:

- Numerical features (BMI, MentHlth, PhysHlth) are scaled with StandardScaler, since they have very different ranges (BMI goes up to 98, the others only to 30).
- Binary features are already 0/1, so we leave them as they are.
- Ordered features (GenHlth, Age, Education, Income) are already stored as codes in the right order (1 = lowest), so we keep them as one column each and scale them like the numerical ones. We did not use one-hot encoding because then the order would be lost.

There are no missing values, so we don't need to fill anything in.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), numerical + ordered),
    ('binary', 'passthrough', binary),
])

The scaler is fitted on TRAIN only when we fit the whole pipeline in Step 5. Here we just test it on TRAIN to check that it works:

In [ ]:
check = pd.DataFrame(preprocessor.fit_transform(X_train), columns=numerical + ordered + binary)
check.describe().round(2)

The scaled features now have mean 0 and std 1, and the binary ones are still 0/1.

One thing to note is that keeping the ordered codes as numbers assumes the steps between them are equal, for example that going from "good" to "fair" health is the same as going from "fair" to "poor". This is not completely true, but we think it is a reasonable simplification.

# Step 5 - Train Logistic Regression

We put the preprocessing from Step 4 and a Logistic Regression model into one Pipeline, and fit the whole Pipeline on TRAIN only.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)                # predicted classes (0 or 1)
y_proba = model.predict_proba(X_test)[:, 1]   # predicted probability of class 1

pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred, 'P(class 1)': y_proba.round(3)}).head()

# Step 6 -  Evaluate the model



# Step 6 - Evaluate on TEST

We compare the predictions from Step 5 with the true classes in TEST. Class 1 (diabetes) is the positive class.

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['0 (no diabetes)', '1 (diabetes)']).plot(cmap='Blues', values_format='d')
plt.title('Confusion matrix on TEST')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True negatives: {tn}, False positives: {fp}, False negatives: {fn}, True positives: {tp}')

In [ ]:
pd.DataFrame({
    'Score': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
    ]
}, index=['Accuracy', 'Precision', 'Recall', 'F1 score']).round(3)